In [1]:
import os
import copy
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
import time

In [ ]:
# Constants / path
batch_size = 32
learning_rate = 0.001
num_epochs = 100
RANDOM_STATE = 1082
criterion = nn.CrossEntropyLoss()
MODEL_NAME = "CNN-LSTM"


In [ ]:
import sys as _sys
from pathlib import Path as _Path

_root = _Path.cwd().resolve()
while not (_root / "eval_metrics.py").is_file() and _root != _root.parent:
    _root = _root.parent
if str(_root) not in _sys.path:
    _sys.path.insert(0, str(_root))
from gps_ids import images

IMAGES_CONFIG = images.TWO_D_IMAGES
device = images.seed_everything()


In [ ]:
IMG = images.loaders(IMAGES_CONFIG, batch_size=batch_size)
dataloaders = IMG.dataloaders
classes = IMG.folder_classes


In [6]:
class CNN_LSTM_Model(nn.Module):
    def __init__(self, num_classes, hidden_dim=128, lstm_layers=1):
        super(CNN_LSTM_Model, self).__init__()
        self.layer1 = self.ConvModule(in_features=1, out_features=64)
        self.layer2 = self.ConvModule(in_features=64, out_features=128)
        self.layer3 = self.ConvModule(in_features=128, out_features=256)
        self.layer4 = self.ConvModule(in_features=256, out_features=512)
        self.gap = nn.AdaptiveAvgPool2d((2, 2))
        self.lstm_input_size = 512 * 2
        self.lstm = nn.LSTM(input_size=self.lstm_input_size, hidden_size=hidden_dim, num_layers=lstm_layers, batch_first=True, bidirectional=False)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 512),
            nn.ReLU(),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        batch_size = x.size(0)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.gap(x)

        x = x.permute(0, 2, 1, 3)
        x = x.contiguous().view(batch_size, 2, -1)
        lstm_out, (h_n, c_n) = self.lstm(x)
        out = lstm_out[:, -1, :]
        out = self.classifier(out)
        return out

    def ConvModule(self, in_features, out_features):
        return nn.Sequential(
            nn.Conv2d(in_channels=in_features, out_channels=out_features, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_features),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )


In [7]:
# Instantiate model, loss, optimizer
model = CNN_LSTM_Model(num_classes=4).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=learning_rate, momentum=0.9)

In [ ]:
# Training loop
best_model_wts = copy.deepcopy(model.state_dict())
best_acc = 0.0
best_epoch = 0

start_time = time.time()

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    all_train_preds = []
    all_train_labels = []
    for inputs, labels in dataloaders['train']:
        inputs = inputs.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
        _, preds = torch.max(outputs, 1)
        all_train_preds.append(preds.cpu().numpy())
        all_train_labels.append(labels.cpu().numpy())
    epoch_loss = running_loss / len(train_dataset)
    train_preds = np.concatenate(all_train_preds)
    train_labels = np.concatenate(all_train_labels)
    train_acc = accuracy_score(train_labels, train_preds)

    # Validation
    model.eval()
    all_val_preds = []
    all_val_labels = []
    with torch.no_grad():
        for inputs, labels in dataloaders['val']:
            inputs = inputs.to(device)
            labels = labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            all_val_preds.append(preds.cpu().numpy())
            all_val_labels.append(labels.cpu().numpy())
    val_preds = np.concatenate(all_val_preds)
    val_labels = np.concatenate(all_val_labels)
    val_acc = accuracy_score(val_labels, val_preds)
    print(f"Epoch {epoch+1}/{num_epochs} - Train Loss: {epoch_loss:.4f} - Train Acc: {train_acc:.4f} - Val Acc: {val_acc:.4f}")

    if val_acc > best_acc:
        best_acc = val_acc
        best_epoch = epoch + 1
        best_model_wts = copy.deepcopy(model.state_dict())

train_time = time.time() - start_time

model.load_state_dict(best_model_wts)

In [ ]:
# Final evaluation
model.eval()
all_preds = []
all_probs = []
all_labels = []
with torch.no_grad():
    for inputs, labels in dataloaders['test']:
        inputs = inputs.to(device)
        labels = labels.to(device)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        all_probs.extend(torch.softmax(outputs, dim=1).cpu().numpy())
        all_preds.append(preds.cpu().numpy())
        all_labels.append(labels.cpu().numpy())
all_preds = np.concatenate(all_preds)
all_labels = np.concatenate(all_labels)

accuracy = accuracy_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds, average='weighted', zero_division=0)
print(f"Test Accuracy: {accuracy * 100:.2f}%")
print(f"Test F1 Score: {f1:.4f}")

In [ ]:
from eval_metrics import evaluate, summary, time_inference_torch
from gps_ids.results import append_validated

_timing = time_inference_torch(model, next(iter(dataloaders['val']))[0].to(device))

row = evaluate(
    all_labels,
    all_preds,
    model_name=MODEL_NAME,
    family="Image",
    y_proba=all_probs,
    train_time_sec=train_time,
    num_epochs=num_epochs,
    parameters=list(dict(model.named_parameters()).keys()),
    timing=_timing,
    **IMG.eval_kwargs(),
)
append_validated(row, IMG.results_csv)
print(summary(row))
print(f"Results appended to {IMG.results_csv}")


In [ ]:
# Save best model
model_save_path = str(images.checkpoint_path(IMAGES_CONFIG, "cnn_lstm"))
checkpoint = {
    'epoch': best_epoch,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict()
}
torch.save(checkpoint, model_save_path)
print(f"Model saved to {model_save_path}")